# Memory Systemsfor Agents

**Module 11 · Lesson 11.4**  
*Netsetos GenAI Engineering Course v2.0*

**In this lesson you will:**
- The Four Kinds of Memory
- Working Memory: the Buffer and its Limit
- Summarization: Compress, Don’t Just Drop
- Semantic Long-Term Memory: Recall by Meaning
- Episodic Memory: Learning From Experience
- Context Engineering: What to Actually Load

---

> This notebook is generated from the published lesson HTML. The HTML is the source of truth - do not hand-edit this notebook. Re-run the generator if the lesson updates.


In [ ]:
# Install Python dependencies used by this lesson (uncomment on Colab / fresh env).
# !pip install numpy langgraph python-dotenv -q  # noqa


In [ ]:
import os
# Load API keys from the environment (or a .env file via python-dotenv).
# Never hardcode keys. Any ONE provider key is enough to start;
# the multi-provider demos are best with two or more.
os.environ.setdefault("OPENAI_API_KEY", "")     # platform.openai.com
os.environ.setdefault("ANTHROPIC_API_KEY", "")    # console.anthropic.com
os.environ.setdefault("GOOGLE_API_KEY", "")     # aistudio.google.com


## The agent that forgets you every turn

> 💡 **Info**
>
> “My name is Priya, my budget is 15000, and I already know Python.” Four turns later the agent asks, “So, what should I call you?” It did not fail to think - it failed to **remember**. An LLM is stateless: the only thing it knows is what is in the context window *right now*, and that window is finite. Real assistants feel like they remember you because engineers bolt on **memory**: a small working set of recent turns, a compressed summary of older ones, a long-term store of facts recalled by meaning, and a log of past experiences it learns from. This lesson builds all four - every mechanism runs with no API key - and then covers the half everyone forgets: **what to actually load** into a context window that is both finite and lossy.

### 🎯 What you will be able to do after this lesson

- **Build** a working-memory buffer, summarization/compaction, semantic vector recall, and episodic memory as small classes.

- **Compare** the four kinds of memory (working / episodic / semantic / procedural) and short-term vs long-term.

- **Operate** context engineering: place facts against “lost in the middle,” compress, and use prompt caching.

- **Evaluate** which memory tool and pattern fits (Mem0 / Zep / Letta / LangMem / LangGraph Store / markdown), and the risks.

> 📦 **Info**
>
> ✅ Before you startThis builds on **11.1** (the in-loop history IS short-term memory in the context window) and **11.3** (a LangGraph checkpointer persists short-term thread state). It also uses **embeddings + cosine similarity** from Module 8 for semantic recall. This lesson is the memory **taxonomy and mechanisms**; graph memory is Lesson 8.7, and multi-agent shared memory is Lesson 11.5.

## 60-Second Warm-Up

Flip each card and answer before revealing.

> 🧠 **Analogy**
>
> **Your own memory is not one system - it is several.** There is the **scratch-pad** you hold a phone number on for ten seconds (working memory), the **diary** of things that happened to you (episodic), the **encyclopedia** of facts you just know (semantic), and the **habits** you do without thinking, like riding a bike (procedural). An agent needs all four, wired from different mechanisms. **Where it breaks down:** a human forgets gracefully and blends these seamlessly; an agent forgets abruptly (a turn falls out of a window) and you must engineer each kind by hand.

> 📦 **Info**
>
> 🚫 Two misconceptions this lesson kills
> - **“A big context window IS memory.”** The window is *working* memory that resets every request. Durable memory is an external store you write to and retrieve from - a context window is not memory.
> - **“Bigger context is always better.”** Effective context is well below the advertised size, and “lost in the middle” means a bloated context can score *worse* than a focused one. What you place, and where, beats raw size.

> 💡 **Info**
>
> ⚠️ Anti-patternDumping every past message into the prompt “so the agent remembers.” It bloats cost, buries the important facts in the middle where attention is lowest, and still overflows on a long conversation. Memory is **selective**: store durably, retrieve narrowly, and place deliberately.

---

## 🎯 Concept 1: The Four Kinds of Memory

### The Four Kinds of Memory

Working, episodic, semantic, procedural. Short-term (in-context) vs long-term (external store). One map for the whole lesson.

Before any code, get the map. Agent memory splits along two axes. By **duration**: **short-term** lives in the context window and resets every request; **long-term** lives in an external store and persists across sessions. By **kind**, the 2026 taxonomy is four: **working** (the running turn, live in context), **episodic** (specific past events and outcomes you can recall), **semantic** (durable facts and preferences), and **procedural** (learned rules and behaviors the agent follows). Every mechanism in this lesson implements one of these, so keep the map in mind: the buffer and summary are *working*-memory management; vector recall is *semantic*; the experience log is *episodic*; and a standing instruction file is *procedural*.

> 📚 **Analogy**
>
> It is the difference between the **sticky note** on your monitor (working: here now, gone tomorrow), your **diary** of what happened (episodic), your mental **encyclopedia** of facts (semantic), and the **habits** in your hands (procedural). You would not store a phone number the way you store how to ride a bike - and an agent should not either.

Where does the standing rule “always answer prices in INR” belong?

**📝 `01_memory_types.py`** - *runnable*

In [ ]:
# THE FOUR KINDS OF AGENT MEMORY - classify what each piece of state IS.
def classify(item):
    kind, span = item["kind"], item["span"]
    if span == "in_context":   return "working",    "the running turn, live in the context window"
    if kind == "fact":         return "semantic",   "a durable fact or preference (long-term store)"
    if kind == "experience":   return "episodic",   "a past event/outcome you can recall (long-term)"
    if kind == "rule":         return "procedural", "a learned instruction the agent follows"
    return "unknown", "?"

items = [
    {"what": "the last 6 chat messages",        "kind": "msgs",       "span": "in_context"},
    {"what": "user prefers evening classes",    "kind": "fact",       "span": "long_term"},
    {"what": "last refund attempt failed - retry with an order id", "kind": "experience", "span": "long_term"},
    {"what": "always answer prices in INR",     "kind": "rule",       "span": "long_term"},
]
for it in items:
    kind, why = classify(it)
    print(f"  {it['what']:44s} -> {kind:11s} ({why})")

# Output:
#   the last 6 chat messages                     -> working     (the running turn, live in the context window)
#   user prefers evening classes                 -> semantic    (a durable fact or preference (long-term store))
#   last refund attempt failed - retry with an order id -> episodic    (a past event/outcome you can recall (long-term))
#   always answer prices in INR                  -> procedural  (a learned instruction the agent follows)

- Anything *in the context window* right now is **working** memory, regardless of content.
- A durable fact or preference is **semantic**; a past event/outcome is **episodic**.
- A learned rule the agent follows is **procedural** - how to behave, not what is true.
- Short-term = in-context (working); the other three are long-term, in an external store.

#### 💡 What just happened

⚡ What just happened?Memory is four kinds along two axes: working / episodic / semantic / procedural, split into short-term (in-context) and long-term (external). Tradeoff / why it matters: each kind needs a different mechanism, so naming the kind tells you which tool to build. The next five steps build one mechanism each.

- Drop each state item into working / episodic / semantic / procedural
- Watch the short-term vs long-term split light up

> 🎬 *Interactive animation in the HTML lesson: **Interactive animation**. Open the HTML version to interact with it.*

---

## 🎯 Concept 2: Working Memory: the Buffer and its Limit

### Working Memory: the Buffer and its Limit

Keep the last N turns; drop the rest. Simple and fast, but it forgets what slides out of a finite window.

The simplest memory is a **buffer**: keep the last N turns and drop the rest. It is fast, needs no store, and is exactly the in-loop history from 11.1. But the context window is **finite**, so a fixed buffer *forgets*: once a turn slides out, it is gone. Below, a window of four turns means “my name is Priya” has already fallen off by the time the user asks “what is my name?” - and no amount of cleverness recovers it, because the information is simply not in the window anymore. Buffers are fine for short exchanges; for anything longer you must either **compress** (Step 3) or **externalize** (Step 4).

> 📝 **Analogy**
>
> It is a **small desk**. Only the last few papers fit; each new one pushes an old one off the edge onto the floor, forgotten. A four-paper desk cannot show you a note you set down six papers ago - it is not hidden, it is on the floor.

A buffer keeps 4 turns. The user said their name 5 turns ago, then asks “what is my name?” What happens?

**📝 `02_buffer_memory.py`** - *runnable*

In [ ]:
# WORKING MEMORY = a sliding window over recent turns. Simple, but it FORGETS what falls out.
class BufferMemory:
    def __init__(self, window):
        self.window, self.turns = window, []
    def add(self, role, text):
        self.turns.append((role, text))
        self.turns = self.turns[-self.window:]     # keep only the last `window` turns
    def context(self):
        return " | ".join(f"{r}:{t}" for r, t in self.turns)

mem = BufferMemory(window=4)
script = [("user", "my name is Priya"), ("user", "I want GenAI"),
          ("user", "my budget is 15000"), ("user", "I know Python"),
          ("user", "what is my name?")]     # the intro has now slid out of a 4-turn window
for role, text in script:
    mem.add(role, text)

print("window =", mem.window, "| turns kept:", len(mem.turns))
print("context now:", mem.context())
answerable = any("name is" in t for _, t in mem.turns)
print("can it answer 'what is my name?' ->", answerable, "(the intro fell out of the window)")

# Output:
# window = 4 | turns kept: 4
# context now: user:I want GenAI | user:my budget is 15000 | user:I know Python | user:what is my name?
# can it answer 'what is my name?' -> False (the intro fell out of the window)

- `add` appends a turn, then slices to the last `window` - older turns are dropped.
- After five turns with window=4, only the last four remain; the intro is gone.
- The check for “name is” is `False` - the fact is not in the window, so it is unanswerable.
- A buffer is working-memory management; it forgets by design, which is why longer chats need more.

#### 💡 What just happened

⚡ What just happened?A buffer is a sliding window over recent turns: fast, stateless, and forgetful. Tradeoff: it caps the tokens you send but loses anything older than the window. When it is not enough: the moment an old detail must survive, compress it (Step 3) or write it to long-term memory (Step 4).

- Slide the window size and feed turns; watch old ones fall out
- See the 'what is my name?' query flip answerable / forgotten

> 🎬 *Interactive animation in the HTML lesson: **Interactive animation**. Open the HTML version to interact with it.*

---

## 🎯 Concept 3: Summarization: Compress, Don’t Just Drop

### Summarization: Compress, Don’t Just Drop

When the buffer overflows, summarize the old turns into a running summary and keep recent ones verbatim. Lossy, but it keeps the key facts.

Dropping old turns loses everything in them; **summarization** (also called compaction) keeps the gist. When the buffer overflows, you compress the oldest turns into a **running summary** and keep only the most recent turns verbatim. The summary carries names, numbers, and preferences forward at a fraction of the tokens. The catch is that summarization is **lossy** - a specific quote or an exact figure can vanish - and re-summarizing naively can drop facts, so a good compactor *carries the prior summary forward* rather than re-deriving it. This is the tiered move production systems make: keep recent turns exact, compress the rest, and only compress when utilization gets high (compaction also breaks **prompt caching** - the cost discount from reusing a stable, unchanged prompt prefix, covered in Step 6 - so you do not do it every turn).

> 📄 **Analogy**
>
> It is **meeting minutes**. You do not keep the full transcript of a two-hour meeting; you write a few lines that keep the decisions, the numbers, and who owns what, and let the small talk go. The minutes are lossy, but the things that matter survive.

The buffer overflows and old turns (name, budget) are compacted into a summary. Later you ask for the name and budget. Result?

**📝 `03_summary_memory.py`** - *runnable*

In [ ]:
# SUMMARIZATION / COMPACTION: instead of dropping old turns, COMPRESS them into a running
# summary and keep only recent turns verbatim. Lossy, but it retains the key facts.
import re
def summarize(prior, turns):              # a scripted extractor (a real system uses an LLM)
    facts = [prior] if prior else []      # carry the running summary forward, then add new facts
    for _, t in turns:
        if "name is" in t:  facts.append("name=" + t.split("name is")[1].strip())
        if "budget" in t:   m = re.search(r"\d+", t); facts.append("budget=" + (m.group() if m else "?"))
        if "Python" in t:   facts.append("knows=Python")
    return "; ".join(dict.fromkeys(facts))   # dedupe, keep order

class SummaryMemory:
    def __init__(self, recent):
        self.recent, self.turns, self.summary = recent, [], ""
    def add(self, role, text):
        self.turns.append((role, text))
        if len(self.turns) > self.recent:                 # overflow -> compact the oldest
            old, self.turns = self.turns[:-self.recent], self.turns[-self.recent:]
            self.summary = summarize(self.summary, old)   # prior summary + facts from the dropped turns

mem = SummaryMemory(recent=2)
for role, text in [("user", "my name is Priya"), ("user", "my budget is 15000"),
                   ("user", "I know Python"), ("user", "recommend a course"),
                   ("user", "what is my name and budget?")]:
    mem.add(role, text)

print("running summary:", mem.summary)
print("recent verbatim:", " | ".join(t for _, t in mem.turns))
print("name+budget still known ->", "Priya" in mem.summary and "15000" in mem.summary)

# Output:
# running summary: name=Priya; budget=15000; knows=Python
# recent verbatim: recommend a course | what is my name and budget?
# name+budget still known -> True

- On overflow, the oldest turns are compressed and only the recent ones stay verbatim.
- `summarize` carries the **prior summary forward**, then adds facts from the dropped turns - so nothing already summarized is lost.
- The running summary keeps `name=Priya; budget=15000; knows=Python` at a fraction of the tokens.
- The name+budget check is `True` - compaction retained the key facts a raw buffer would have dropped.

#### 💡 What just happened

⚡ What just happened?Summarization compresses old turns into a running summary while keeping recent turns verbatim. Tradeoff: it is lossy (exact quotes/numbers can vanish) and it breaks prompt caching, so compact on a threshold, not every turn. Carry the prior summary forward so re-summarizing does not quietly drop facts.

- Overflow the buffer; toggle drop vs summarize
- Watch the running summary keep name + budget while raw turns compress

> 🎬 *Interactive animation in the HTML lesson: **Interactive animation**. Open the HTML version to interact with it.*

---

## 🎯 Concept 4: Semantic Long-Term Memory: Recall by Meaning

### Semantic Long-Term Memory: Recall by Meaning

Extract facts, embed them, retrieve by similarity. This is how an assistant remembers you across sessions.

Buffers and summaries are still short-term - they live in one conversation. **Semantic long-term memory** is the external store that survives across sessions. The mechanism is the retrieval you know from Module 8, now used as memory: on the **write** path you decide what is worth remembering, extract it as a clean fact, and **embed** it into a vector; on the **read** path you embed the query and return the facts whose vectors are most **similar** (cosine). Days later, in a brand-new conversation, “what is her budget?” retrieves “budget is 15000” by meaning, not exact words. This similarity-based recall is the core of how assistants like ChatGPT, Claude, and Gemini remember you across sessions - each layering its own fact extraction and injection on top. The **write** path also **consolidates**: when a fact changes (her budget rises to 20000), a near-duplicate should *update* the old entry, not pile up a second, contradictory one - that is how you keep memory from going stale. (The demo uses a *toy* bag-of-words embedder so it runs with no key; production swaps in a real embedding model - the mechanism is identical.)

> 📚 **Analogy**
>
> It is a **librarian who files by meaning**. You do not remember the shelf number; you ask “do we have anything on budgets?” and she walks straight to the right books - because she filed them by topic, not by the exact title. Embeddings are that topic-based filing.

You stored “Priya’s budget is 15000 rupees” last week. Today, in a new session, you ask “what can she spend?” How is it found?

**📝 `04_vector_memory.py`** - *runnable*

In [ ]:
# SEMANTIC LONG-TERM MEMORY: extract facts, EMBED them, recall by similarity - across sessions.
# The WRITE path also CONSOLIDATES: a near-duplicate UPDATES the old fact so memory does not go stale.
import numpy as np
VOCAB = ["python","developer","hyderabad","budget","15000","20000","rupees",
         "evening","classes","genai","name","priya"]
def embed(text):                          # TOY bag-of-words embedder (prod: a real embedding model)
    t = text.lower()
    v = np.array([1.0 if w in t else 0.0 for w in VOCAB])
    n = np.linalg.norm(v)
    return v / n if n else v

class VectorMemory:
    def __init__(self): self.facts = []
    def store(self, fact, thresh=0.7):
        fe = embed(fact)
        for i, (old, oe) in enumerate(self.facts):
            if float(np.dot(fe, oe)) >= thresh:   # same topic -> UPDATE (avoid stale duplicates)
                self.facts[i] = (fact, fe); return
        self.facts.append((fact, fe))             # new topic -> APPEND
    def recall(self, query, k=1):
        qe = embed(query)
        ranked = sorted(self.facts, key=lambda f: -float(np.dot(qe, f[1])))
        return [f[0] for f in ranked[:k]]

mem = VectorMemory()
for fact in ["Priya is a Python developer in Hyderabad",
             "Priya's budget is 15000 rupees",
             "Priya prefers evening classes"]:
    mem.store(fact)                       # written once; recallable days later, in a new session

for q in ["what is her budget in rupees?", "when does she like classes?", "is she a Python developer?"]:
    print(f"  '{q}' -> {mem.recall(q)}")

mem.store("Priya's budget is now 20000 rupees")   # a near-duplicate UPDATES the budget, not appends
print("after an update, facts stored:", len(mem.facts), "->", mem.recall("what is her budget?"))

# Output:
#   'what is her budget in rupees?' -> ["Priya's budget is 15000 rupees"]
#   'when does she like classes?' -> ['Priya prefers evening classes']
#   'is she a Python developer?' -> ['Priya is a Python developer in Hyderabad']
# after an update, facts stored: 3 -> ["Priya's budget is now 20000 rupees"]

- `store` embeds each fact into a vector - the **write** path - and *consolidates*: a near-duplicate (the new budget) UPDATES the old fact instead of piling up a stale duplicate.
- `recall` embeds the query and ranks facts by cosine similarity - the **read** path.
- Each query retrieves the right fact by *meaning* (budget, classes, developer), not exact words.
- After the budget update there are still **3** facts and recall returns the current 20000 - consolidation is how you keep memory from going stale. The embedder is a toy; swap in a real model and the code is production semantic memory.

#### 💡 What just happened

⚡ What just happened?Semantic memory = extract a fact, embed it, and recall by similarity - long-term, cross-session, exactly the retrieval from Module 8 used as memory. Tradeoff: both the write path (what to store) and the read path (what to retrieve and how much) are engineering decisions, not automatic - store too much and recall gets noisy.

- Store facts as vectors, then type a query
- Watch cosine similarity rank and retrieve the top fact

> 🎬 *Interactive animation in the HTML lesson: **Interactive animation**. Open the HTML version to interact with it.*

---

## 🎯 Concept 5: Episodic Memory: Learning From Experience

### Episodic Memory: Learning From Experience

Store past attempts and their outcomes with a reflection. On a similar task, recall it and avoid repeating the mistake.

Semantic memory stores *facts*; **episodic memory** stores *experiences* - what the agent tried, what happened, and a short **reflection**. On a later, similar task the agent retrieves those reflections and adapts, so it stops repeating the same failure. This is the **Reflexion** pattern from 11.2, now as a memory system: after a failed refund the agent writes “a refund needs the order id up front,” and the next refund task pulls that reflection and asks for the id first. It is how an agent improves *without retraining* - through structured recall of its own past, not gradient updates. (Semantic = what is true; episodic = what happened; procedural = the rule you distil from it.)

> 🔬 **Analogy**
>
> It is a **lab notebook**. You write down what you tried and what went wrong - ‘the reaction failed because the flask was wet.’ Next time a similar experiment comes up, you read your own note and dry the flask first. The agent reads its own past attempts the same way.

An agent failed a refund last week and logged a reflection. A new refund task arrives. What does episodic memory give it?

**📝 `05_episodic_memory.py`** - *runnable*

In [ ]:
# EPISODIC MEMORY: store past ATTEMPTS (task, outcome, reflection) and recall them on a similar
# task, to learn from failure (Reflexion). A recurring lesson is then PROMOTED to a PROCEDURAL rule.
class EpisodicMemory:
    def __init__(self): self.episodes, self.rules = [], []
    def record(self, task, outcome, reflection):
        self.episodes.append({"task": task, "outcome": outcome, "reflection": reflection})
    def advice_for(self, task):
        key = task.split()[0].lower()      # crude task-type match (prod: embed the task)
        hits = [e for e in self.episodes if e["task"].split()[0].lower() == key]
        return [e["reflection"] for e in hits if e["outcome"] == "failed"]
    def promote(self, task):               # distil a recalled lesson into a durable procedural rule
        for r in self.advice_for(task):
            rule = "RULE: " + r
            if rule not in self.rules: self.rules.append(rule)

mem = EpisodicMemory()
mem.record("refund order #A1", "failed", "a refund needs the order id up front - ask for it first")
mem.record("refund order #B2", "success", "asked for the order id, then refunded")

new_task = "refund order #C3"
advice = mem.advice_for(new_task)          # EPISODIC recall
print("new task:", new_task)
print("recalled reflections:", advice)
print("acts differently this time ->", bool(advice))
mem.promote(new_task)                       # EPISODIC -> PROCEDURAL: the lesson becomes a standing rule
print("promoted to procedural rules:", mem.rules)

# Output:
# new task: refund order #C3
# recalled reflections: ['a refund needs the order id up front - ask for it first']
# acts differently this time -> True
# promoted to procedural rules: ['RULE: a refund needs the order id up front - ask for it first']

- `record` stores an episode: the task, its outcome, and a short reflection.
- `advice_for` matches the task type and returns the reflections from *failed* attempts.
- The new refund task recalls “ask for the order id first” - a lesson from a past failure.
- `promote` then distils that recalled lesson into a durable **procedural** rule - episodic experience becoming a standing behavior, with no retraining.

#### 💡 What just happened

⚡ What just happened?Episodic memory stores attempts + outcomes + reflections and recalls them on similar tasks - the Reflexion loop as a durable memory. Tradeoff / when to use it: it shines on repeated task types where past failures are informative; it adds storage and retrieval you must curate so stale or wrong lessons do not mislead.

- Run a task, record success or failure + a reflection
- A similar new task retrieves the reflection and avoids the mistake

> 🎬 *Interactive animation in the HTML lesson: **Interactive animation**. Open the HTML version to interact with it.*

---

## 🎯 Concept 6: Context Engineering: What to Actually Load

### Context Engineering: What to Actually Load

The context window is finite and lossy. Retrieval is half the job; selecting, compressing, and placing is the other half.

You can store perfect memories and still fail, because the context window is **finite** and **lossy**. Models attend most to the **start and end** of the context and under-attend the **middle** - the “lost in the middle” effect - so effective usable context is well below the advertised size, and stuffing everything in can score *worse* than a focused subset. Retrieval (Step 4) is only half the job; **context engineering** is the other half: **write** durable memory outside the window, **select** only what is relevant, **compress** the rest (Step 3), and **isolate** per task - then **place** the facts that matter at the edges. **Prompt caching** cuts the cost of a stable prefix, but note it is at odds with compaction: the moment you rewrite the prefix, you break the cache.

> 🗃️ **Analogy**
>
> It is a **tidy desk versus a cluttered one**. Pile every paper on and the one that matters gets buried in the middle of the stack, where your eye never lands - you skim the top and the bottom. Lay out only the few papers you need, the important one on top, and you find it instantly. Context is that desk.

You must put one critical fact where the model will actually use it. Best position in a long context?

**📝 `06_context_engineering.py`** - *runnable*

In [ ]:
# CONTEXT ENGINEERING: retrieval is only half. The context window is finite AND lossy -
# models attend to the START and END, not the MIDDLE ("lost in the middle"). Placement matters.
def recall_score(position, n):            # a MODEL of the U-shaped attention curve (illustrative)
    rel = position / (n - 1)              # 0.0 = first, 1.0 = last
    edge = min(rel, 1 - rel)              # distance from the nearer edge (0 at edges, 0.5 in middle)
    return round(1.0 - 1.4 * edge, 2)     # high at the edges, dips in the middle

n = 11
print("needle position -> recall (illustrative U-shape):")
for pos in (0, n // 2, n - 1):
    where = {0: "START", n // 2: "MIDDLE", n - 1: "END"}[pos]
    print(f"  {where:6s} (pos {pos:2d}/{n-1}) -> {recall_score(pos, n)}")

best = max(range(n), key=lambda p: recall_score(p, n))
print("fix: put the critical fact at the edges; a focused context beats a bloated one.")
print("best positions -> the edges (pos 0 and pos", n - 1, "), worst -> the middle")

# Output:
# needle position -> recall (illustrative U-shape):
#   START  (pos  0/10) -> 1.0
#   MIDDLE (pos  5/10) -> 0.3
#   END    (pos 10/10) -> 1.0
# fix: put the critical fact at the edges; a focused context beats a bloated one.
# best positions -> the edges (pos 0 and pos 10 ), worst -> the middle

- `recall_score` models the U-shape: high at the edges, dipping toward the middle (illustrative).
- A needle at the START or END scores far higher than the same needle in the MIDDLE.
- The fix is placement: put the facts that matter at the edges, and keep the context focused.
- This is why retrieval is not enough - you must select, compress, and place, not just fetch.

#### 💡 What just happened

⚡ What just happened?Context engineering is the read-side discipline: write, select, compress, isolate - then place critical facts at the edges to beat lost-in-the-middle. Tradeoff: prompt caching rewards a stable prefix but compaction rewrites it, so the two pull against each other - cache the stable system context, compact the volatile history.

- Drag a needle fact across the context window
- A U-shaped recall meter dips in the middle; compare bloated vs focused

> 🎬 *Interactive animation in the HTML lesson: **Interactive animation**. Open the HTML version to interact with it.*

---

## 🎯 Concept 7: Memory in Practice: Frameworks, Homes, and Risks

### Memory in Practice: Frameworks, Homes, and Risks

Where each memory lives, which tool does it, the tiered pattern, and why memory can be dangerous.

You have built the mechanisms; here is how production wires them. Each memory has a **home**: working memory in the **context**; semantic facts in a **vector store**; episodes in an **experience log**; procedural rules in a **markdown file** in source control (`CLAUDE.md` / `AGENTS.md`). Frameworks package the extract-embed-recall you wrote by hand: **Mem0** (passive fact extraction, personalization), **Zep** (temporal reasoning over a knowledge graph), **Letta/MemGPT** (self-editing memory blocks), **LangMem** (semantic/episodic/procedural for LangChain). In LangGraph, the **checkpointer** is within-thread short-term (11.3) and the **Store** is cross-thread long-term, namespaced per user. The dominant production pattern is **tiered**: a small always-in-context core + a vector-retrieval layer + an explicit forgetting policy. And memory is not free of risk: **memory poisoning persists across sessions** (unlike a one-shot prompt injection), stale or contradictory memories mislead, and personal data in embeddings is hard to delete - a real compliance gap.

> 📁 **Analogy**
>
> It is your **filing system at home**: the sticky note on the monitor (context), the filing cabinet (vector store), the lab notebook (episodic log), and the house-rules poster on the fridge (`CLAUDE.md`). You would not keep the electricity bill on a sticky note or your daily to-do in the cabinet - each memory goes to the home that fits it.

A user invokes their right to erasure. Which unit do you delete in a namespaced memory store?

**📝 `07_store.py`** - *illustrative*

In [ ]:
# WHERE LONG-TERM MEMORY LIVES: a cross-thread STORE (LangGraph) or a memory service (Mem0).
from langgraph.store.memory import InMemoryStore   # cross-thread long-term store (prod: Postgres / Redis)
store = InMemoryStore()
ns = ("user-123", "facts")                          # a namespace isolates memory per user
store.put(ns, "budget", {"text": "budget is 15000 INR"})
store.put(ns, "prefers", {"text": "evening classes"})
hits = store.search(ns, query="what is the budget?")    # semantic search across one user's memories
print([h.value["text"] for h in hits])

# Or a managed service (Mem0) does the extract + store + recall for you:
from mem0 import Memory
mem = Memory()
mem.add("My budget is 15000 and I prefer evenings", user_id="priya")   # extracts + stores the facts
print(mem.search("budget?", user_id="priya"))

# Output: (illustrative minimal example - needs `pip install langgraph mem0ai` + a key for Mem0.)
#   LangGraph's Store is the cross-thread long-term home; the checkpointer from 11.3 is within-thread.
#   Mem0 / Zep / Letta / LangMem wrap the same extract -> embed -> recall you built by hand. A namespace
#   isolates one user's memory - the unit you delete for a right-to-erasure (DPDP) request.

- LangGraph’s `Store` is the cross-thread long-term home; the checkpointer (11.3) is within-thread.
- A **namespace** per user isolates memory - and is the unit you delete for a right-to-erasure request.
- **Mem0 / Zep / Letta / LangMem** wrap the same extract → embed → recall you built by hand.
- Choose by need: Mem0 for personalization, Zep for temporal reasoning, Letta for self-editing, LangMem/Store in the LangChain stack.

#### 💡 What just happened

⚡ What just happened?Each memory kind has a home (context, vector store, experience log, markdown), frameworks package the mechanisms, and the pattern is tiered (core + retrieval + forgetting). Tradeoff / risk: memory poisoning persists across sessions and stale memories mislead, so treat memory as attack surface - validate what you write, scope it per user, and make it deletable.

- Route each memory to its home: context / vector store / episodic log / CLAUDE.md
- See tiered hot / warm / cold and a memory-poisoning flag

> 🎬 *Interactive animation in the HTML lesson: **Interactive animation**. Open the HTML version to interact with it.*

## Putting It Together

> ✅ **Info**
>
> 🧠 The whole picture
> Agent memory is **four kinds** - working, episodic, semantic, procedural - split into short-term (in-context) and long-term (external) (Step 1). A **buffer** is working memory: fast, but it forgets what slides out of a finite window (Step 2). **Summarization** compresses old turns into a running summary and keeps recent ones verbatim - lossy but retentive (Step 3). **Semantic memory** extracts facts, embeds them, and recalls by similarity across sessions (Step 4). **Episodic memory** stores attempts and reflections so the agent learns from failure (Step 5). But storing is not enough: **context engineering** decides what reaches a finite, lossy window - select, compress, and place at the edges (Step 6). And in production each memory has a home, a framework, and real risks like poisoning (Step 7). Memory is what turns a stateless model into an agent that knows you.

| Kind | Holds | Duration | Mechanism in this lesson |
|---|---|---|---|
| Working | the running turn | short-term (in context) | buffer + summarization (Steps 2–3) |
| Semantic | durable facts / preferences | long-term (store) | vector recall (Step 4) |
| Episodic | past attempts + outcomes | long-term (store) | experience log (Step 5) |
| Procedural | learned rules / behaviors | long-term (markdown) | CLAUDE.md / AGENTS.md (Step 7) |

> 📦 **Info**
>
> ➡️ Where this goes nextYou can now give an agent memory. Next: **multi-agent** systems in Lesson 11.5 need *shared* memory across agents; **human-in-the-loop** in Lesson 11.10 uses persisted state for approvals; and **graph memory** for multi-hop and temporal recall is Lesson 8.7 (GraphRAG). The references are [Lost in the Middle](https://arxiv.org/abs/2307.03172), the [Mem0](https://github.com/mem0ai/mem0) and [Letta](https://docs.letta.com) projects, and Anthropic’s [Building Effective Agents](https://www.anthropic.com/research/building-effective-agents).

*Practice exercises are in the companion practice notebook.*

---

## 🎓 Summary

You've completed the practical part of **Memory Systemsfor Agents**.

### Next steps:
1. Re-run cells with different parameters to build intuition
2. Try the practice exercises (see `practice-lab/practice-lab-11_4.ipynb` if available)
3. Review the full HTML lesson for the complete narrative

*Generated from `lesson-11.4.html` - regenerate this notebook whenever the source HTML is updated.*
